# SQL reference 2: NULLs and conditional logic

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-reference-02-nulls-and-conditional-logic.ipynb)

Three-valued logic, `IS DISTINCT FROM`, the `NOT IN` trap, `COALESCE`/`NULLIF`, NULLs in strings and aggregates, ANSI mode and the `try_` functions, and conditional expressions with `CASE`, `IF` and `DECODE`. Practise it in [SQL practice 2: NULLs and conditional logic](sql-practice-02-nulls-and-conditional-logic.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0 (Spark SQL).*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local Spark session, the shared tables as temporary views: `customers`, `orders`, `order_items`, `products`, `events` and `employees` (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry, two pairs of employees tie on salary, Dana out-earns her manager and Jo has no department), and three helpers: `show(query)` runs a query and prints its rows untruncated, `plan(query)` returns what `EXPLAIN` prints, and `expected_error()` prints the error class of an example that fails on purpose.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
# Shared sample tables as temporary views. Each statement is ordinary Spark SQL.
TABLES = {
    "customers": """CREATE OR REPLACE TEMP VIEW customers AS SELECT * FROM VALUES
  (1, 'Acme Corp',      'North', DATE'2024-01-03'),
  (2, 'Bright Retail',  'North', DATE'2024-02-11'),
  (3, 'Cyan Traders',   'South', DATE'2024-03-15'),
  (4, 'Delta Grocers',  NULL,    DATE'2024-05-02'),   -- region never arrived from the CRM
  (5, 'Echo Supplies',  'West',  DATE'2024-06-20'),
  (6, 'Foxtrot Foods',  'West',  DATE'2024-08-01'),
  (7, 'Gamma Sports',   'East',  DATE'2024-09-14'),   -- 7 and 8 have never ordered
  (8, 'Hotel Homeware', 'East',  DATE'2024-11-05')
AS customers(customer_id, name, region, signup_date)""",
    "orders": """CREATE OR REPLACE TEMP VIEW orders AS SELECT order_id, customer_id, status, order_ts, CAST(amount AS DECIMAL(12,2)) AS amount FROM VALUES
  (1,  1, 'paid',     TIMESTAMP'2026-01-05 09:15:00', 120.00),
  (2,  1, 'shipped',  TIMESTAMP'2026-01-05 10:05:00', 84.50),
  (3,  1, 'returned', TIMESTAMP'2026-01-05 11:40:00', 30.00),
  (4,  2, 'paid',     TIMESTAMP'2026-01-06 15:20:00', 260.00),
  (5,  2, 'placed',   TIMESTAMP'2026-01-07 08:05:00', 55.25),
  (6,  3, 'shipped',  TIMESTAMP'2026-01-07 17:45:00', 149.00),
  (7,  3, 'paid',     TIMESTAMP'2026-01-09 12:10:00', 310.00),
  (8,  4, 'returned', TIMESTAMP'2026-01-10 19:30:00', 89.90),
  (9,  4, 'shipped',  TIMESTAMP'2026-01-12 08:55:00', 1180.00),
  (10, 5, 'paid',     TIMESTAMP'2026-01-12 13:00:00', 74.00),
  (11, 6, 'placed',   TIMESTAMP'2026-01-13 10:25:00', 210.00),
  (12, 6, 'shipped',  TIMESTAMP'2026-01-14 00:30:00', 96.40)   -- just after midnight UTC
AS orders(order_id, customer_id, status, order_ts, amount)""",
    "order_items": """CREATE OR REPLACE TEMP VIEW order_items AS SELECT order_id, sku, qty, CAST(unit_price AS DECIMAL(8,2)) AS unit_price FROM VALUES
  (1, 'SKU-4001', 2, 12.50), (1, 'SKU-4007', 1, 95.00), (2, 'SKU-4002', 1, 34.50), (2, 'SKU-4009', 2, 25.00),
  (3, 'SKU-4002', 1, 30.00), (4, 'SKU-4003', 1, 149.00), (4, 'SKU-4004', 2, 55.50), (5, 'SKU-4005', 1, 55.25),
  (6, 'SKU-4003', 1, 149.00), (7, 'SKU-4006', 1, 189.00), (7, 'SKU-4008', 1, 121.00), (8, 'SKU-4010', 1, 89.90),
  (9, 'SKU-4011', 20, 59.00), (10, 'SKU-4012', 2, 37.00), (11, 'SKU-4013', 1, 210.00),
  (12, 'SKU-4014', 1, 41.40), (12, 'SKU-4005', 1, 55.00)
AS order_items(order_id, sku, qty, unit_price)""",
    "products": """CREATE OR REPLACE TEMP VIEW products AS SELECT sku, category, CAST(list_price AS DECIMAL(8,2)) AS list_price FROM VALUES
  ('SKU-4001', 'Kitchen', 12.50), ('SKU-4002', 'Kitchen', 34.50), ('SKU-4003', 'Garden', 149.00),
  ('SKU-4004', 'Garden', 55.50), ('SKU-4005', 'Pantry', 55.25), ('SKU-4006', 'Outdoor', 189.00),
  ('SKU-4007', 'Kitchen', 95.00), ('SKU-4008', 'Outdoor', 121.00), ('SKU-4009', 'Pantry', 25.00),
  ('SKU-4010', 'Garden', 89.90), ('SKU-4011', 'Pantry', 59.00), ('SKU-4012', 'Kitchen', 37.00),
  ('SKU-4013', 'Outdoor', 210.00)   -- SKU-4014 has no product row
AS products(sku, category, list_price)""",
    "events": """CREATE OR REPLACE TEMP VIEW events AS SELECT * FROM VALUES
  (1, 'view', TIMESTAMP'2026-01-06 09:00:00'), (1, 'add_to_cart', TIMESTAMP'2026-01-06 09:02:00'),
  (1, 'purchase', TIMESTAMP'2026-01-06 09:10:00'), (2, 'view', TIMESTAMP'2026-01-07 10:00:00'),
  (2, 'purchase', TIMESTAMP'2026-01-07 10:40:00'), (3, 'view', TIMESTAMP'2026-01-06 11:15:00'),
  (3, 'view', TIMESTAMP'2026-01-08 11:20:00'), (4, 'add_to_cart', TIMESTAMP'2026-01-09 20:05:00'),
  (4, 'purchase', TIMESTAMP'2026-01-10 08:30:00'), (5, 'view', TIMESTAMP'2026-01-11 14:45:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00'), (5, 'purchase', TIMESTAMP'2026-01-12 12:30:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00')   -- an SDK retry committed this event twice
AS events(user_id, event, ts)""",
    "employees": """CREATE OR REPLACE TEMP VIEW employees AS SELECT * FROM VALUES
  (1,  'Ada',    'Engineering', CAST(NULL AS INT), 185000, DATE'2018-03-01'),
  (2,  'Brett',  'Engineering', 1,    150000, DATE'2019-07-15'),
  (3,  'Chen',   'Engineering', 1,    150000, DATE'2020-01-06'),
  (4,  'Dana',   'Engineering', 2,    155000, DATE'2022-09-12'),   -- out-earns her manager
  (5,  'Eli',    'Sales',       1,    130000, DATE'2019-02-01'),
  (6,  'Farah',  'Sales',       5,    95000,  DATE'2021-05-17'),
  (7,  'Gus',    'Sales',       5,    95000,  DATE'2023-01-09'),
  (8,  'Hana',   'Finance',     1,    140000, DATE'2017-11-20'),
  (9,  'Ivan',   'Finance',     8,    88000,  DATE'2024-04-01'),
  (10, 'Jo',     NULL,          8,    70000,  DATE'2025-06-30')   -- department not assigned yet
AS employees(emp_id, name, dept, manager_id, salary, hire_date)""",
}
for statement in TABLES.values():
    spark.sql(statement)

import contextlib
import io
import logging
import re


def show(query, n=50, **args):
    """Run a query and print up to n rows without truncating values. Keyword arguments bind :name parameters."""
    spark.sql(query, args=args or None).show(n, truncate=False)


def plan(query, mode="formatted"):
    """The plan EXPLAIN prints for a query, as text."""
    return spark.sql(f"EXPLAIN {mode.upper()} {query}").first()[0]


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:120])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

## Contents

| section | entries |
| --- | --- |
| [NULL semantics](#null-semantics) | [Three-valued logic](#three-valued-logic), [`IS NULL`, `IS DISTINCT FROM` and `<=>`](#is-null-is-distinct-from-and-), [NULL with `IN` and `NOT IN`](#null-with-in-and-not-in) |
| [Replacing and producing NULLs](#replacing-and-producing-nulls) | [`COALESCE`, `IFNULL`/`NVL`, `NULLIF`, `NVL2`](#coalesce-ifnullnvl-nullif-nvl2), [NULLs in arithmetic, strings and aggregates](#nulls-in-arithmetic-strings-and-aggregates), [Division by zero and ANSI mode](#division-by-zero-and-ansi-mode) |
| [Conditional expressions](#conditional-expressions) | [`CASE`](#case), [`IF`, `DECODE` and boolean expressions](#if-decode-and-boolean-expressions) |

## NULL semantics

### Three-valued logic

NULL means "unknown". Comparisons and arithmetic involving NULL produce NULL, and boolean logic has a third value,
UNKNOWN (written NULL):

| `a` | `b` | `a AND b` | `a OR b` | `NOT a` |
| --- | --- | --- | --- | --- |
| TRUE | NULL | NULL | TRUE | FALSE |
| FALSE | NULL | FALSE | NULL | TRUE |
| NULL | NULL | NULL | NULL | NULL |

`WHERE`, `HAVING`, `ON` and `CASE WHEN` keep a row only when the condition is **TRUE**; NULL is treated like FALSE.
That is why `WHERE region <> 'North'` silently drops rows whose region is NULL.

**Notes:** `x = NULL` is always NULL, never TRUE; test with `IS NULL`. `IS TRUE`, `IS NOT TRUE`, `IS FALSE` turn
UNKNOWN into a definite answer: `flag IS NOT TRUE` keeps both FALSE and NULL.

In [ ]:
show("SELECT NULL = NULL AS eq, NULL AND FALSE AS and_false, NULL OR TRUE AS or_true, 1 + NULL AS plus, NULL IS NULL AS is_null")
show("SELECT count(*) AS not_north FROM customers WHERE region <> 'North'")                     # customer 4 is dropped
show("SELECT count(*) AS not_north FROM customers WHERE region <> 'North' OR region IS NULL")

### `IS NULL`, `IS DISTINCT FROM` and `<=>`

| predicate | NULL vs value | NULL vs NULL | use |
| --- | --- | --- | --- |
| `a = b` | NULL | NULL | ordinary equality |
| `a IS NULL` / `IS NOT NULL` | | | test for missing values |
| `a IS NOT DISTINCT FROM b` / `a <=> b` | FALSE | TRUE | null-safe equality: comparing nullable keys, change detection |
| `a IS DISTINCT FROM b` | TRUE | FALSE | "has the value changed?" including to or from NULL |

**Notes:** `<=>` is Spark and MySQL syntax; `IS [NOT] DISTINCT FROM` is standard and works in Spark, Postgres,
Snowflake, BigQuery and DuckDB. Joining on nullable keys with `=` drops NULL keys; use `<=>` only when NULL really
should match NULL.

In [ ]:
show("""
SELECT name, dept,
       dept = 'Engineering'                AS eq,
       dept IS DISTINCT FROM 'Engineering' AS changed,
       dept <=> NULL                       AS null_safe_null
FROM employees WHERE emp_id IN (1, 5, 10) ORDER BY emp_id
""")

### NULL with `IN` and `NOT IN`

`x IN (a, b, NULL)` is TRUE when `x` matches, otherwise NULL (never FALSE). So `x NOT IN (a, b, NULL)` is never TRUE
and returns **no rows**. The same happens with `NOT IN (SELECT col ...)` when the subquery returns any NULL.

**Notes:** prefer `NOT EXISTS` (or a left anti join) for "rows with no match": it is NULL-safe and usually planned as
an anti join ([subqueries](sql-05-aggregations-and-ctes.ipynb)).

In [ ]:
show("SELECT customer_id FROM customers WHERE customer_id NOT IN (1, 2, 3, 4, 5, 6) ORDER BY 1")
show("SELECT customer_id FROM customers WHERE customer_id NOT IN (1, 2, 3, 4, 5, 6, NULL) ORDER BY 1")   # no rows
show("""
SELECT c.customer_id FROM customers c
WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)
ORDER BY 1
""")

## Replacing and producing NULLs

### `COALESCE`, `IFNULL`/`NVL`, `NULLIF`, `NVL2`

| function | returns |
| --- | --- |
| `coalesce(a, b, ...)` | the first non-NULL argument (NULL if all are NULL); evaluates arguments left to right |
| `ifnull(a, b)` / `nvl(a, b)` | two-argument `coalesce` |
| `nullif(a, b)` | NULL if `a = b`, else `a`; the standard guard against division by zero: `x / nullif(y, 0)` |
| `nvl2(a, if_not_null, if_null)` | choose by whether `a` is NULL |

**Notes:** all arguments must share a common type (`coalesce(amount, 'n/a')` mixes decimal and string: cast
explicitly). Replacing NULL with 0 before averaging changes the answer: decide whether "unknown" means zero.

In [ ]:
show("SELECT customer_id, coalesce(region, 'Unknown') AS region, nvl2(region, 'known', 'missing') AS state FROM customers WHERE customer_id BETWEEN 3 AND 5 ORDER BY 1")
show("SELECT 10 / nullif(0, 0) AS safe, coalesce(NULL, NULL, 'third') AS first_non_null")

### NULLs in arithmetic, strings and aggregates

| expression | with a NULL input |
| --- | --- |
| `a + b`, `a * b`, `a \|\| b`, `concat(a, b)` | NULL |
| `concat_ws(sep, a, b, ...)` | skips NULL arguments |
| `greatest(a, b)`, `least(a, b)` | Spark, Postgres and DuckDB ignore NULLs; Snowflake, BigQuery and MySQL return NULL |
| `count(*)` | counts rows, including all-NULL rows |
| `count(col)`, `sum`, `avg`, `min`, `max` | ignore NULLs; `sum`/`avg`/`min`/`max` of no non-NULL values is NULL |

**Notes:** `avg(col)` divides by the number of **non-NULL** values. `sum` over an empty set is NULL, not 0: wrap it
in `coalesce(sum(x), 0)` when reporting.

In [ ]:
show("SELECT name || ' (' || region || ')' AS pipe_concat, concat_ws(' / ', name, region) AS ws FROM customers WHERE customer_id IN (3, 4) ORDER BY customer_id")
show("SELECT greatest(1, NULL, 3) AS g, least(NULL, 2) AS l")
show("""
SELECT count(*) AS rows, count(dept) AS with_dept, count(DISTINCT dept) AS depts,
       sum(CASE WHEN dept IS NULL THEN salary END) AS unassigned_salary,
       sum(CASE WHEN dept = 'Marketing' THEN salary END) AS marketing_salary   -- no rows: NULL, not 0
FROM employees
""")

### Division by zero and ANSI mode

Spark 4.0 runs in **ANSI mode** by default (`spark.sql.ansi.enabled=true`): division by zero, overflow and invalid
casts raise errors instead of silently returning NULL. Use the `try_` functions when NULL is the result you want.

| function | returns NULL instead of failing on |
| --- | --- |
| `try_divide(a, b)` | division by zero |
| `try_cast(x AS type)` | an invalid cast (`'abc'` to `INT`) |
| `try_add`, `try_subtract`, `try_multiply` | overflow |
| `try_element_at(array_or_map, key)` | a missing index or key |

**Notes:** Postgres and Snowflake also raise on division by zero; MySQL returns NULL. `nullif(y, 0)` is the portable
guard.

In [ ]:
with expected_error():
    show("SELECT 1 / 0 AS boom")
show("SELECT try_divide(1, 0) AS a, 1 / nullif(0, 0) AS b, try_cast('12.5x' AS DECIMAL(10,2)) AS c")

## Conditional expressions

### `CASE`

```sql
CASE WHEN cond1 THEN r1 [WHEN cond2 THEN r2 ...] [ELSE rn] END      -- searched
CASE expr WHEN v1 THEN r1 [WHEN v2 THEN r2 ...] [ELSE rn] END       -- simple
```

| behaviour | detail |
| --- | --- |
| evaluation | branches are tested in order; the first TRUE wins, so order overlapping ranges from most to least specific |
| no match | `ELSE` is optional and defaults to NULL |
| NULL tests | a simple `CASE x WHEN NULL` never matches (it compares with `=`); use `CASE WHEN x IS NULL` |
| result type | all branches are coerced to a common type |

**Use it for:** bucketing, mapping codes to labels, conditional aggregation (`sum(CASE WHEN ... THEN amount END)`),
custom sort orders, and guarding expressions.

In [ ]:
show("""
SELECT order_id, amount,
       CASE WHEN amount >= 500 THEN 'large'
            WHEN amount >= 100 THEN 'medium'
            ELSE 'small' END AS size,
       CASE status WHEN 'placed' THEN 1 WHEN 'paid' THEN 2 WHEN 'shipped' THEN 3 ELSE -1 END AS stage
FROM orders WHERE order_id <= 5 ORDER BY order_id
""")
show("""
SELECT name, CASE region WHEN NULL THEN 'never matches' ELSE coalesce(region, 'unknown') END AS region_label
FROM customers WHERE customer_id = 4
""")
show("SELECT order_id, status FROM orders ORDER BY CASE status WHEN 'placed' THEN 0 WHEN 'paid' THEN 1 ELSE 2 END, order_id LIMIT 5")

### `IF`, `DECODE` and boolean expressions

| function | equivalent |
| --- | --- |
| `if(cond, a, b)` / `iff` (Snowflake) | `CASE WHEN cond THEN a ELSE b END` |
| `decode(x, v1, r1, v2, r2, ..., default)` | simple `CASE` (Oracle-style; matches NULL to NULL in Spark) |
| a boolean expression as a column | `amount > 100 AS is_big` |
| `count_if(cond)` | `count(CASE WHEN cond THEN 1 END)` |

In [ ]:
show("SELECT order_id, if(amount > 100, 'big', 'small') AS size, decode(status, 'paid', 'P', 'shipped', 'S', '?') AS code, amount > 100 AS is_big FROM orders WHERE order_id <= 4 ORDER BY 1")
show("SELECT count_if(amount > 100) AS big_orders, count_if(status = 'returned') AS returns FROM orders")

**Next:** practise these entries in [SQL practice 2: NULLs and conditional logic](sql-practice-02-nulls-and-conditional-logic.ipynb).